In [3]:
import pandas as pd

df = pd.read_csv(
    '../data/food.csv.gz',
    sep='\t',                  # TAB separated, not comma!
    compression='gzip',
    nrows=500000,              # Only load first 500k rows
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())

(500000, 210)
['code', 'url', 'creator', 'created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'last_modified_by', 'last_updated_t', 'last_updated_datetime', 'product_name', 'abbreviated_product_name', 'generic_name', 'quantity', 'packaging', 'packaging_tags', 'packaging_en', 'packaging_text', 'brands', 'brands_tags', 'brands_en', 'categories', 'categories_tags', 'categories_en', 'origins', 'origins_tags', 'origins_en', 'manufacturing_places', 'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_en', 'emb_codes', 'emb_codes_tags', 'first_packaging_code_geo', 'cities', 'cities_tags', 'purchase_places', 'stores', 'countries', 'countries_tags', 'countries_en', 'ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'allergens', 'allergens_en', 'traces', 'traces_tags', 'traces_en', 'serving_size', 'serving_quantity', 'no_nutrition_data', 'additives_n', 'additives', 'additives_tags', 'additives_en', 'nutriscore_score', 'nutriscore_grade', 'nova_

In [4]:
# Find our key columns
sugar_cols = [col for col in df.columns if 'sugar' in col.lower()]
protein_cols = [col for col in df.columns if 'protein' in col.lower()]
fat_cols = [col for col in df.columns if 'fat' in col.lower()]
fiber_cols = [col for col in df.columns if 'fiber' in col.lower()]
category_cols = [col for col in df.columns if 'categor' in col.lower()]

print("Sugar columns:", sugar_cols)
print("Protein columns:", protein_cols)
print("Fat columns:", fat_cols)
print("Fiber columns:", fiber_cols)
print("Category columns:", category_cols)

Sugar columns: ['sugars_100g', 'added-sugars_100g']
Protein columns: ['proteins_100g', 'serum-proteins_100g', 'collagen-meat-protein-ratio_100g']
Fat columns: ['energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'trans-fat_100g']
Fiber columns: ['fiber_100g', 'soluble-fiber_100g', 'insoluble-fiber_100g']
Category columns: ['categories', 'categories_tags', 'categories_en', 'main_category', 'main_category_en']


In [5]:
# Preview the key columns we expect to use
key_cols = ['product_name', 'categories_tags', 'ingredients_text']
print(df[key_cols].head(3))

                    product_name categories_tags  \
0  Limonade artisanale a la rose             NaN   
1                  M&amp;M white             NaN   
2                   Chocolate n3             NaN   

                                    ingredients_text  
0                                                NaN  
1  Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...  
2                                                NaN  


In [6]:
# Check how much data is missing in our key columns
key_cols = ['product_name', 'categories_tags', 'ingredients_text', 
            'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']

missing = df[key_cols].isnull().sum()
total = len(df)

print("Missing values:\n")
for col, count in missing.items():
    pct = (count / total) * 100
    print(f"  {col}: {count} missing ({pct:.1f}%)")

Missing values:

  product_name: 15690 missing (3.1%)
  categories_tags: 231471 missing (46.3%)
  ingredients_text: 232286 missing (46.5%)
  sugars_100g: 394381 missing (78.9%)
  proteins_100g: 391135 missing (78.2%)
  fat_100g: 391276 missing (78.3%)
  fiber_100g: 422247 missing (84.4%)


In [7]:
# ============================================
# STORY 1: DATA CLEANING
# ============================================

print("Shape before cleaning:", df.shape)

# Step 1: Keep only the columns we actually need
cols_to_keep = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g'
]
df_clean = df[cols_to_keep].copy()

# Step 2: Drop rows missing product_name, sugars, or proteins
# (these are essential for our analysis)
df_clean = df_clean.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

print("Shape after dropping missing essentials:", df_clean.shape)

# Step 3: Remove biologically impossible values
# (per 100g, nothing can exceed 100g)
df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100) &
    (df_clean['fat_100g'].isna() | ((df_clean['fat_100g'] >= 0) & (df_clean['fat_100g'] <= 100)))
]

print("Shape after removing impossible values:", df_clean.shape)

# Step 4: Reset index
df_clean = df_clean.reset_index(drop=True)

print("\n✅ Cleaning complete!")
print("Final clean dataset shape:", df_clean.shape)
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum())

Shape before cleaning: (500000, 210)
Shape after dropping missing essentials: (103859, 7)
Shape after removing impossible values: (103682, 7)

✅ Cleaning complete!
Final clean dataset shape: (103682, 7)

Missing values in clean dataset:
product_name            0
categories_tags     53632
ingredients_text    58882
sugars_100g             0
proteins_100g           0
fat_100g              207
fiber_100g          28169
dtype: int64


In [8]:
print("Shape before cleaning:", df.shape)

cols_to_keep = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g'
]
df_clean = df[cols_to_keep].copy()

df_clean = df_clean.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

print("Shape after dropping missing essentials:", df_clean.shape)

df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100) &
    (df_clean['fat_100g'].isna() | ((df_clean['fat_100g'] >= 0) & (df_clean['fat_100g'] <= 100)))
]

print("Shape after removing impossible values:", df_clean.shape)

df_clean = df_clean.reset_index(drop=True)

print("\n✅ Cleaning complete!")
print("Final clean dataset shape:", df_clean.shape)
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum())

Shape before cleaning: (500000, 210)
Shape after dropping missing essentials: (103859, 7)
Shape after removing impossible values: (103682, 7)

✅ Cleaning complete!
Final clean dataset shape: (103682, 7)

Missing values in clean dataset:
product_name            0
categories_tags     53632
ingredients_text    58882
sugars_100g             0
proteins_100g           0
fat_100g              207
fiber_100g          28169
dtype: int64
